<a href="https://colab.research.google.com/github/Varshith246/GenAI-Healthcare-Chatbot/blob/main/MediAssist_AI_Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The patient has high blood sugar.",
    "The glucose level is elevated.",
    "The patient has a fever."
]

embeddings = model.encode(sentences)
print("Shape:", embeddings.shape)

sim_0_1 = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
sim_0_2 = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]

print("blood sugar vs glucose:", round(float(sim_0_1), 4))
print("blood sugar vs fever:  ", round(float(sim_0_2), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape: (3, 384)
blood sugar vs glucose: 0.6806
blood sugar vs fever:   0.5388


In [ ]:

# STEP 1: Install required libraries
!pip install -q sentence-transformers faiss-cpu

# STEP 2: Import libraries
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# STEP 3: Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# STEP 4: Create medical documents
medical_documents = [
    "Hemoglobin carries oxygen throughout the human body.",
    "Blood glucose measures the amount of sugar in the blood.",
    "White blood cells help the body fight infections.",
    "Platelets help in blood clotting.",
    "Cholesterol is a fatty substance present in the blood.",
    "Creatinine is a waste product used to assess kidney function.",
    "Red blood cells transport oxygen from the lungs to body tissues.",
    "Insulin helps regulate blood glucose levels."
]

# STEP 5: Convert documents into embeddings
doc_embeddings = model.encode(medical_documents)

# Convert embeddings into float32 (required by FAISS)
doc_embeddings = np.array(doc_embeddings).astype("float32")

# Normalize embeddings for cosine similarity
faiss.normalize_L2(doc_embeddings)

# STEP 6: Create FAISS index
dimension = doc_embeddings.shape[1]

medical_index = faiss.IndexFlatIP(dimension)

# STEP 7: Add document embeddings to FAISS
medical_index.add(doc_embeddings)

print("Total documents stored:", medical_index.ntotal)


# STEP 8: Create semantic search function
def search_medical_info(question, top_k=2):

    # Convert user question into embedding
    query_embedding = model.encode([question])

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    # Normalize query embedding
    faiss.normalize_L2(query_embedding)

    # Search in FAISS
    scores, indices = medical_index.search(
        query_embedding, top_k
    )

    # Retrieve original documents using indices
    results = []

    for i in indices[0]:
        if i != -1:
            results.append(medical_documents[i])

    return scores[0], indices[0], results


# STEP 9: Test the search function
question = "Which blood cells help fight infections?"

scores, indices, results = search_medical_info(question, top_k=2)

print("\nUser Question:", question)

print("\nFAISS Scores:", scores)

print("\nFAISS Indices:", indices)

print("\nRetrieved Documents:")

for document in results:
    print("-", document)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total documents stored: 8

User Question: Which blood cells help fight infections?

FAISS Scores: [0.79470354 0.44956094]

FAISS Indices: [2 6]

Retrieved Documents:
- White blood cells help the body fight infections.
- Red blood cells transport oxygen from the lungs to body tissues.
